# Project — Airline AI Assistant (trợ lý AI hàng không)

Bây giờ chúng ta ghép những gì đã học để làm AI Customer Support (hỗ trợ khách hàng) cho một hãng hàng không.

In [1]:
# imports (nhập thư viện)

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [4]:
# Khởi tạo

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key tồn tại và bắt đầu bằng {openai_api_key[:8]}")
else:
    print("OpenAI API Key chưa được đặt")
    
MODEL = "gpt-4.1-mini"
openai = OpenAI()

# Thay thế: nếu muốn dùng Ollama thay OpenAI
# Kiểm tra Ollama đang chạy local (xem bài tập week1/day2) rồi bỏ comment 2 dòng sau
# MODEL = "llama3.2"
# openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')


OpenAI API Key tồn tại và bắt đầu bằng sk-proj-


In [5]:
system_message = """
Bạn là trợ lý hữu ích của hãng hàng không FlightAI.
Trả lời ngắn, lịch sự, không quá 1 câu.
Luôn chính xác. Nếu không biết đáp án, hãy nói vậy.
"""

In [6]:
# Chatbot Gradio chưa có tools — chỉ trả lời từ system_message

def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## Tools (công cụ)

Tools là tính năng cực mạnh do các frontier LLMs cung cấp.

Với tools, bạn viết một function, rồi để LLM gọi function đó như một phần của response.

Nghe hơi ma: chúng ta cho nó quyền chạy code trên máy mình?

Ừ, kiểu vậy.

In [7]:
# Bắt đầu bằng một function hữu ích

ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}

def get_ticket_price(destination_city):
    print(f"Tool được gọi cho thành phố {destination_city}")
    price = ticket_prices.get(destination_city.lower(), "Không rõ giá vé")
    return f"Giá vé khứ hồi đến {destination_city} là {price}"


In [8]:
get_ticket_price("London")

Tool được gọi cho thành phố London


'Giá vé khứ hồi đến London là $799'

In [9]:
# Cần một cấu trúc dictionary đặc biệt để mô tả function của chúng ta:

price_function = {
    "name": "get_ticket_price",
    "description": "Lấy giá vé khứ hồi đến thành phố đích. Truyền tên thành phố bằng tiếng Anh (ví dụ: London, Paris, Tokyo, Berlin).",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "Thành phố khách muốn đến, viết bằng tiếng Anh",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

In [10]:
# Và dictionary này được đưa vào list tools:

tools = [{"type": "function", "function": price_function}]

In [11]:
tools

[{'type': 'function',
  'function': {'name': 'get_ticket_price',
   'description': 'Lấy giá vé khứ hồi đến thành phố đích. Truyền tên thành phố bằng tiếng Anh (ví dụ: London, Paris, Tokyo, Berlin).',
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
      'description': 'Thành phố khách muốn đến, viết bằng tiếng Anh'}},
    'required': ['destination_city'],
    'additionalProperties': False}}}]

## Để OpenAI dùng Tool của chúng ta

Có vài bước hơi rườm để cho OpenAI "gọi tool".

Thực tế chúng ta cho LLM cơ hội **báo** rằng nó muốn chúng ta chạy tool.

Function chat mới trông như sau:

In [12]:
# Nếu model yêu cầu tool (finish_reason=="tool_calls") thì chạy tool rồi gọi lại LLM

def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        response = handle_tool_call(message)
        messages.append(message)
        messages.append(response)
        response = openai.chat.completions.create(model=MODEL, messages=messages)
    
    return response.choices[0].message.content

In [13]:
# Phải viết function handle_tool_call:

def handle_tool_call(message):
    tool_call = message.tool_calls[0]
    if tool_call.function.name == "get_ticket_price":
        arguments = json.loads(tool_call.function.arguments)
        city = arguments.get('destination_city')
        price_details = get_ticket_price(city)
        response = {
            "role": "tool",
            "content": price_details,
            "tool_call_id": tool_call.id
        }
    return response

In [14]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Tool được gọi cho thành phố London


## Thêm vài cải thiện

Xử lý nhiều tool calls trong 1 response

Xử lý nhiều tool calls nối tiếp nhau

In [15]:
# Nhiều tool trong cùng 1 lượt: handle_tool_calls trả về list rồi extend vào messages

def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages)
    
    return response.choices[0].message.content

In [16]:
# Xử lý mọi tool_call trong cùng một message (không chỉ cái đầu)

def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses

In [18]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


In [19]:
# Vòng while: model có thể gọi tool nhiều lượt liên tiếp trước khi trả lời user

def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    return response.choices[0].message.content

In [20]:
# sqlite3: database (cơ sở dữ liệu) file, không cần server riêng

import sqlite3


In [21]:
# Tạo bảng prices nếu chưa có: city là khóa chính, price là số

DB = "prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)')
    conn.commit()

In [22]:
def get_ticket_price(city):
    print(f"DATABASE TOOL ĐƯỢC GỌI: Lấy giá cho {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Giá vé đến {city} là ${result[0]}" if result else "Không có dữ liệu giá cho thành phố này"

In [23]:
get_ticket_price("London")

DATABASE TOOL ĐƯỢC GỌI: Lấy giá cho London


'Không có dữ liệu giá cho thành phố này'

In [24]:
# Ghi hoặc cập nhật giá vé trong database

def set_ticket_price(city, price):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?', (city.lower(), price, price))
        conn.commit()

In [25]:
# Nạp giá mẫu vào database

ticket_prices = {"london":799, "paris": 899, "tokyo": 1420, "sydney": 2999}
for city, price in ticket_prices.items():
    set_ticket_price(city, price)

In [26]:
get_ticket_price("Tokyo")

DATABASE TOOL ĐƯỢC GỌI: Lấy giá cho Tokyo


'Giá vé đến Tokyo là $1420.0'

In [27]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


## Bài tập

Hãy thêm một tool để **đặt giá vé** (set the price of a ticket)!

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Ứng dụng kinh doanh</h2>
            <span style="color:#181;">Hy vọng điều này không cần phải nói thêm! Bạn đã có khả năng giao actions (hành động) cho LLMs. Airline Assistant này giờ không chỉ trả lời câu hỏi — nó có thể tương tác với booking APIs để đặt chỗ!</span>
        </td>
    </tr>
</table>